## **ClinRAG: Clinical Knowledge Assistant**

### **RAG-Based Medical Question Answering System**
This implementation use vector DB which for knowladge base and do the key based search on the knowladge base and send the context to LLM for generationg a response.


## **System Workflow (Step-by-Step)**
**Step 1: Data Preparation – Extract and clean Wikipedia text.**

**Step 2: Chunking – Split into smaller segments with overlap.**

**Step 3: Embedding Creation – Convert chunks into vectors.**

**Step 4: Store in Vector DB – Save embeddings with metadata.**

**Step 5: User Query Processing – Convert question to embedding.**

**Step 6: Similarity Search – Retrieve top relevant chunks.**

**Step 7: Answer Generation – Send retrieved context + question to LLM.**


In [1]:
#Install dependency to load text files

! pip install langchain-community --quiet


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Step 1: Data Preparation – Extract and clean Wikipedia text.

from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import TextLoader

loader = DirectoryLoader('clinical_data', glob="*.txt", show_progress=True, loader_cls=TextLoader)

clinical_docs = loader.load()

100%|█████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 652.23it/s]


In [3]:
# Step 2: Chunking – Split into smaller segments with overlap.

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

clinical_doc_chunks = text_splitter.split_documents(clinical_docs)

print("Number of Documents:", len(clinical_docs))
print("Number of Chunks:", len(clinical_doc_chunks))


In [8]:
! pip install langchain-chroma --quiet
# ! pip install langchain-openai
! pip install langchain sentence-transformers --quiet


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
# Step 3: Embedding Creation – Convert chunks into vectors.
from langchain_chroma import Chroma
#from langchain_openai import OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

#f = open("keys/.openai_api_key.txt")
#OPENAI_API_KEY = f.read()
#embedding_model = OpenAIEmbeddings(api_key=OPENAI_API_KEY)


embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

# Initialize the database connection
# If database exist, it will connect with the collection_name and persist_directory
# Otherwise a new collection will be created
db = Chroma(collection_name="vector_database", 
            embedding_function=embedding_model, 
            persist_directory="./chroma_db_")

# Step 4: Store in Vector DB – Save embeddings with metadata.
db.add_documents(documents=clinical_doc_chunks)


C:\Users\skumar7\AppData\Local\Temp\ipykernel_30816\3425810933.py:21: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

D:\AITestLabs\HybridRAG\.env_hybridrag\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\skumar7\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
# Step 1: Initialize the Chroma DB Connection

from langchain_chroma import Chroma

# Initialize the database connection
# If database exist, it will connect with the collection_name and persist_directory
# Otherwise a new collection will be created
db = Chroma(collection_name="vector_database", 
            embedding_function=embedding_model, 
            persist_directory="./chroma_db_")

# We can check the already existing values
print(len(db.get()["ids"]))

181


In [14]:
# Step 5: User Query Processing – Convert question to embedding.

from langchain_core.prompts import ChatPromptTemplate

PROMPT_TEMPLATE = """You are helpful AI Assistant.
Answer the question based only on the following context: {context}
Answer the question based on the above context: {question}.
Provide a detailed answer.
Don’t justify your answers.
Don’t give information not mentioned in the CONTEXT INFORMATION.
Do not say "according to the context" or "mentioned in the context" or similar.
"""

prompt_template = ChatPromptTemplate(
    messages=[
        PROMPT_TEMPLATE
    ]
)

# prompt_template

In [13]:
# Step 6: Similarity Search – Retrieve top relevant chunks.

retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})


In [27]:
! pip install langchain_groq --quiet


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
# Step 7: Answer Generation

#from langchain_openai import ChatOpenAI

#chat_model = ChatOpenAI(api_key=OPENAI_API_KEY)


from langchain_groq import ChatGroq

GROQ_API_KEY = "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"
chat_model = ChatGroq(
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.1-8b-instant",
)


In [29]:
# Initialize a Output Parser

from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [30]:
#chain prompt template chat model and output parser

generator_chain = prompt_template | chat_model | parser


In [31]:
# Helper function to join the retrieved chunks

def format_docs(retrived_kb_docs):
    return "\n\n".join(kb_doc.page_content for kb_doc in retrived_kb_docs)
    

In [32]:
# Define a RAG Chain with knowladge base and to pass question

from langchain_core.runnables import RunnablePassthrough

rag_chain = {
    "context": retriever | format_docs, 
    "question": RunnablePassthrough()
} | generator_chain


In [33]:
# 1. What is Alzheimer’s disease?
query = 'What is Alzheimer’s disease?'
rag_chain.invoke(query)

"Alzheimer's disease is a neurodegenerative disease and the most common form of dementia, accounting for around 60â€“70% of cases. It is named after German psychiatrist and pathologist Alois Alzheimer, who first described it in 1906."

In [35]:
# 2. What type of disorder is Alzheimer’s disease classified as?
query = 'What type of disorder is Alzheimer’s disease classified as?'
rag_chain.invoke(query)

"Alzheimer's disease is classified as a neurodegenerative disease."

In [36]:
# 3. What is the primary characteristic of Alzheimer’s disease?
query = 'What is the primary characteristic of Alzheimer’s disease?'
rag_chain.invoke(query)

"The primary characteristic of Alzheimer's disease is a progressive pattern of cognitive and functional impairment."

In [37]:
# 4. Is Alzheimer’s disease reversible?
query = 'Is Alzheimer’s disease reversible?'
rag_chain.invoke(query)

"No, Alzheimer's disease is not reversible. The text states that with Alzheimer's disease, people go through the reverse process of progressive cognitive impairment, implying that the progression of the disease cannot be reversed."

In [38]:
# 5. What is the most common cause of dementia?
query = 'What is the most common cause of dementia?'
rag_chain.invoke(query)

"The most common cause of dementia is Alzheimer's disease, accounting for around 60–70% of cases."

In [39]:
# 6. What is typically the earliest symptom of Alzheimer’s disease?
query = 'What is typically the earliest symptom of Alzheimer’s disease?'
rag_chain.invoke(query)

"The first symptom of Alzheimer's disease is short-term memory loss, which shows up as difficulty in remembering recently learned facts and inability to acquire new information."

In [40]:
# 7. How does Alzheimer’s disease affect memory over time?
query = 'How does Alzheimer’s disease affect memory over time?'
rag_chain.invoke(query)

"Alzheimer's disease affects memory over time by leading to increasing impairment of learning and memory. The most common early symptom is difficulty in remembering recent events. \n\nOlder memories of the person's life (episodic memory), facts learned (semantic memory), and implicit memory (the memory of the body) are affected differently. \n\nAs the disease advances, difficulties with remembering recent events become more pronounced. Eventually, the increasing impairment of learning and memory leads to a definitive diagnosis."

In [41]:
# 8. Besides memory, what other cognitive functions are affected?
query = 'Besides memory, what other cognitive functions are affected?'
rag_chain.invoke(query)

"Besides memory, difficulties with language and executive functions are affected in people with Alzheimer's disease. Additionally, some individuals may experience:\n\n1. Agnosia: problems with perception.\n2. Apraxia: difficulties with executing movements."

In [42]:
# 9. How does Alzheimer’s disease affect behavior?
query = 'How does Alzheimer’s disease affect behavior?'
rag_chain.invoke(query)

"Alzheimer's disease affects behavior in several ways. \n\nBehavioral and neuropsychiatric changes become more prevalent as the disease advances. \nCommon manifestations include wandering, irritability, emotional liability, leading to crying, outbursts of unpremeditated aggression, or resistance to caregiving.\nSundowning can also appear.\nApproximately 30% of people with Alzheimer's disease develop illusionary misidentifications and other delusional symptoms.\nSubjects also lose insight of their disease process and limitations (anosognosia).\nUrinary incontinence can develop."

In [43]:
# 10. What happens in the advanced stages of Alzheimer’s disease?
query = 'What happens in the advanced stages of Alzheimer’s disease?'
rag_chain.invoke(query)

"In the advanced stages of Alzheimer's disease:\n\n- There is complete dependence on caregivers.\n- Language is reduced to simple phrases or even single words.\n- Eventually, there is a complete loss of speech.\n- Despite the loss of verbal language abilities, people can often understand and return emotional signals.\n- Aggressiveness can still be present.\n- Extreme apathy and exhaustion are much more common symptoms.\n- People with Alzheimer's disease will ultimately not be able to perform."

In [44]:
# 11. What are amyloid plaques?
query = 'What are amyloid plaques?'
rag_chain.invoke(query)

'Amyloid plaques are dense, mostly insoluble deposits of amyloid beta peptide and cellular material outside and around neurons.'

In [45]:
# 12. What are neurofibrillary tangles?
query = 'What are neurofibrillary tangles?'
rag_chain.invoke(query)

'Neurofibrillary tangles are aggregates of the microtubule-associated protein tau which has become hyperphosphorylated and accumulates inside neurons.'

In [46]:
# 13. What role does beta-amyloid play in Alzheimer’s disease?
query = 'What role does beta-amyloid play in Alzheimer’s disease?'
rag_chain.invoke(query)

"Beta-amyloid plays a crucial role in the formation of amyloid plaques in Alzheimer's disease. It is a fragment produced by enzymes acting on the amyloid-beta precursor protein, and it is essential in the progression of the disease."

In [47]:
# 14. What role does tau protein play in Alzheimer’s disease?
query = 'What role does tau protein play in Alzheimer’s disease?'
rag_chain.invoke(query)

"The tau protein plays a key role in Alzheimer's disease by:\n\n1. Initiating the disease cascade: Abnormalities of the tau protein are proposed to initiate the disease cascade, at least in cases of idiopathic Alzheimer's disease.\n\n2. Hyperphosphorylation: The tau protein undergoes hyperphosphorylation, a chemical change that occurs in specific vulnerable neuronal populations.\n\n3. Pairing with other threads: The hyperphosphorylated tau protein pairs with other threads, creating neurofibrillary tangles.\n\n4. Disintegrating the neuron's transport system: The creation of neurofibrillary tangles disrupts the neuron's transport system.\n\n5. Causing neuronal death: Pathogenic tau can cause neuronal death through transposable element dysregulation and necroptosis."

In [48]:
# 15. How does Alzheimer’s disease affect brain cells?
query = 'How does Alzheimer’s disease affect brain cells?'
rag_chain.invoke(query)

"Alzheimer's disease affects brain cells in several ways:\n\n1. **Abnormal aggregation of tau protein**: The disease is a tauopathy due to the abnormal aggregation of the tau protein within cells, leading to destabilization of the cytoskeleton.\n\n2. **Damage to microtubules**: The tau protein, which stabilizes microtubules when phosphorylated, fails to do so, causing damage to the microtubules in the cytoskeleton of neurons.\n\n3. **Loss of neurons and synapses**: The disease leads to the disappearance of neurons and their synapses, which is a prominent correlate of dementia.\n\n4. **Inflammation**: Alzheimer's disease causes inflammation in the brain, mediating neuroinflammation through cells like microglia, astrocytes, oligodendrocytes, lymphocytes, and myeloid cells.\n\n5. **Selectively vulnerable neurons**: The disease selectively affects certain neurons and synapses, leaving others spared, which is a characteristic feature of Alzheimer's disease."

In [49]:
# 16. What is the biggest risk factor for Alzheimer’s disease?
query = 'What is the biggest risk factor for Alzheimer’s disease?'
rag_chain.invoke(query)

"The strongest genetic risk factor for Alzheimer's disease is the APOEÎµ4 allele. \n\nThe APOEÎµ4 allele disrupts the function of apolipoprotein E, which plays a major role in lipid-binding proteins in lipoprotein particles. \n\nBetween 40% and 80% of people with Alzheimer's disease possess at least one APOEÎµ4 allele. \n\nThe APOEÎµ4 allele increases the risk of the disease by three times in heterozygotes and by 15 times in homozygotes."

In [50]:
# 17. Does genetics influence Alzheimer’s disease?
query = 'Does genetics influence Alzheimer’s disease?'
rag_chain.invoke(query)

"Yes, genetics influences Alzheimer's disease.\n\nAlzheimer's disease is substantially polygenic, which means it is influenced by many genes.\n\nAutosomal dominant mutations cause 1-2% of Alzheimer's cases, resulting in early-onset familial Alzheimer's disease.\n\nThe strongest genetic risk factor for sporadic Alzheimer's disease is APOEε4.\n\nBetween 40% and 80% of people with Alzheimer's disease possess at least one APOEε4 allele.\n\nThe APOEε4 allele increases the risk of the disease by three times in heterozygotes and by 15 times in homozygotes.\n\nEarly-onset Alzheimer's is about 90% heritable.\n\nFamilial Alzheimer's disease usually implies two or more persons affected in a family.\n\nLess than 5% of sporadic Alzheimer's disease have an earlier onset."

In [51]:
# 18. What is the role of the APOE gene?
query = 'What is the role of the APOE gene?'
rag_chain.invoke(query)

"The APOE gene plays a major role in lipid-binding proteins in lipoprotein particles. It disrupts its function when it has the Îµ4 allele, which is a major genetic risk factor for Alzheimer's disease."

In [52]:
# 19. Can lifestyle factors influence Alzheimer’s risk?
query = 'Can lifestyle factors influence Alzheimer’s risk?'
rag_chain.invoke(query)

"Yes, lifestyle factors can influence Alzheimer's risk. \n\nDiet is a modifiable risk factor for the development of Alzheimer's disease. \nSmoking is a significant Alzheimer's disease risk factor.\nExposure to air pollution may be a contributing factor to the development of Alzheimer's disease.\nIntellectual activities such as playing chess or regular social interaction have been linked to a reduced risk of Alzheimer's disease.\n\nThese lifestyle factors can contribute to the development of Alzheimer's disease or reduce the risk. However, the exact relationship between these factors and Alzheimer's disease is still not fully understood and requires further research."

In [53]:
# 20. Is family history a risk factor?
query = 'Is family history a risk factor?'
rag_chain.invoke(query)

"Family history is implied to be a risk factor for Alzheimer's disease, as the progression of the disease usually involves two or more persons affected in one or more generations."

In [54]:
# 21. How is Alzheimer’s disease diagnosed?
query = 'How is Alzheimer’s disease diagnosed?'
rag_chain.invoke(query)

"Alzheimer's disease diagnosis involves the following steps:\n\n1. Assessment of intellectual functioning, including memory testing, to further characterise the state of the disease.\n2. Medical organizations' diagnostic criteria to ease and standardise the diagnostic process for practising physicians.\n3. Post-mortem evaluations when brain material is available and can be examined histologically for senile plaques and neurofibrillary tangles to confirm a definitive diagnosis.\n\nAdditionally, the following are crucial in the diagnosis of Alzheimer's disease:\n\n1. Further neurological examinations to differential diagnose Alzheimer's disease and other diseases.\n2. Interviews with family members to assess the person's condition.\n3. Caregivers' viewpoints to provide information on daily living abilities and the decrease in the person's mental function.\n4. Cognitive tests, such as the mini–mental state examination (MMSE), to help in the diagnosis of Alzheimer's disease. This test invo

In [55]:
# 22. Is there a single definitive test for Alzheimer’s?
query = 'Is there a single definitive test for Alzheimer’s?'
rag_chain.invoke(query)

"There is no single definitive test for Alzheimer's. Definitive diagnosis can only be confirmed with post-mortem evaluations when brain material is available and can be examined histologically for senile plaques and neurofibrillary tangles."

In [56]:
# 23. What imaging techniques are used?
query = 'What imaging techniques are used?'
rag_chain.invoke(query)

'The imaging techniques used are:\n\n1. Computed Tomography (CT)\n2. Magnetic Resonance Imaging (MRI)\n3. Single-photon emission computed tomography (SPECT)\n4. Positron emission tomography (PET)\n5. FDG-PET scan'

In [57]:
# 24. Can Alzheimer’s be confirmed definitively?
query = 'Can Alzheimer’s be confirmed definitively?'
rag_chain.invoke(query)

"Definitive diagnosis of Alzheimer's disease can only be confirmed with post-mortem evaluations when brain material is available and can be examined histologically for senile plaques and neurofibrillary tangles."

In [58]:
# 25. What role do biomarkers play?
query = 'What role do biomarkers play?'
rag_chain.invoke(query)

'Biomarkers specific for AD are required. Further research is needed to determine these biomarkers.'

In [59]:
# 26. Is there a cure for Alzheimer’s disease?
query = 'Is there a cure for Alzheimer’s disease?'
rag_chain.invoke(query)

"There is no cure for Alzheimer's disease."

In [60]:
# 27. What is the goal of current treatments?
query = 'What is the goal of current treatments?'
rag_chain.invoke(query)

'The goal of current treatments is to improve behavior, mood, and function, although to a lesser extent.'

In [61]:
# 28. What types of medications are used?
query = 'What types of medications are used?'
rag_chain.invoke(query)

"There are two types of medications used to treat the cognitive symptoms of Alzheimer's disease (AD):\n\n1. Acetylcholinesterase inhibitors:\n   - Tacrine\n   - Rivastigmine\n   - Galantamine\n   - Donepezil\n\n2. Memantine:\n   - An NMDA receptor antagonist"

In [62]:
# 29. How do cholinesterase inhibitors work?
query = 'How do cholinesterase inhibitors work?'
rag_chain.invoke(query)

'Cholinesterase inhibitors work by reducing the rate at which the body breaks down acetylcholine (ACh). This increases the concentration of ACh in the brain. This is done by inhibiting acetylcholinesterase, the enzyme responsible for breaking down ACh.'

In [63]:
# 30. What supportive care is important?
query = 'What supportive care is important?'
rag_chain.invoke(query)

"Supportive care that is important includes:\n\n1. Adherence to simplified routines\n2. Placing of safety locks\n3. Labeling of household items to cue the person with the disease\n4. Use of modified daily life objects\n5. Preparing food in smaller pieces or puréeing it when eating becomes problematic\n6. Using a feeding aid when swallowing difficulties arise\n7. A healthy diet\n8. Physical activity\n9. Social engagement\n\nThese supportive care measures can increase safety, reduce caretaker burden, and relieve discomfort in people with Alzheimer's disease."

In [64]:
# 31. Can Alzheimer’s disease be prevented?
query = 'Can Alzheimer’s disease be prevented?'
rag_chain.invoke(query)

"There is no evidence that supports any particular measure in preventing Alzheimer's disease. However, a healthy diet, physical activity, and social engagement may help in reducing the risk of cognitive decline and Alzheimer's."

In [65]:
# 32. What lifestyle factors may reduce risk?
query = 'What lifestyle factors may reduce risk?'
rag_chain.invoke(query)

"Healthy diet, physical activity, and social engagement are generally beneficial in aging, and may help in reducing the risk of cognitive decline and Alzheimer's. \n\nPhysical activity is associated with a lower risk of AD, specifically the WHO recommendations for physical activity."

In [66]:
# 33. Does education level influence risk?
query = 'Does education level influence risk?'
rag_chain.invoke(query)

"Yes, education level influences risk. Higher education contributes to a reduced risk of developing AD, or of delaying the onset of symptoms. It delays the onset of Alzheimer's disease syndrome without changing the duration of survival."

In [67]:
# 34. How common is Alzheimer’s disease globally?
query = 'How common is Alzheimer’s disease globally?'
rag_chain.invoke(query)

"As of 2020, there were approximately 50 million people worldwide with Alzheimer's disease."

In [68]:
# 35. Does prevalence increase with age?
query = 'Does prevalence increase with age?'
rag_chain.invoke(query)

'Yes, prevalence increases with age.\n\nPrevalence depends on the mean age of the population. \n\nIn the United States in 2020, the prevalence of AD was estimated to be:\n- 5.3% for those in the 60–74 age group\n- 13.8% in the 74–84 group\n- 34.6% in those greater than 85\n\nThe prevalence rate is estimated to triple by 2050, reaching 152 million.'

In [69]:
# 36. Is Alzheimer’s more common in women or men?
query = 'Is Alzheimer’s more common in women or men?'
rag_chain.invoke(query)

"Women are more commonly affected by Alzheimer's disease. \n\nThere are higher age-adjusted numbers for women, and even with the same amount of AD pathology observed in a woman compared to a man, there is greater cognitive decline in a woman."

In [70]:
# 37. How does Alzheimer’s disease progress?
query = 'How does Alzheimer’s disease progress?'
rag_chain.invoke(query)

"Alzheimer's disease progresses in a three-stage process:\n\n1. **Early or Mild Stage**: \n   - The disease targets the hippocampus.\n   - The first symptoms are memory impairment.\n\n2. **Middle or Moderate Stage**: \n   - The degree of memory impairment increases.\n   - Gradually, other bodily functions are lost.\n\n3. **Late or Severe Stage**: \n   - The progression ultimately leads to death.\n   - This stage is reached after the early and middle stages.\n\nThe progression of the disease can vary in speed, but the average life expectancy following diagnosis is three to twelve years."

In [71]:
# 38. What are the general stages?
query = 'What are the general stages?'
rag_chain.invoke(query)

'There are three general stages: \n\n1. Early stage\n2. Middle stage\n3. Late stage'

In [72]:
# 39. What happens in the late stage?
query = 'What happens in the late stage?'
rag_chain.invoke(query)

'Late stage: \n\nIn the late stage, there is no information provided regarding what happens.'

In [73]:
# 40. Who first described Alzheimer’s disease?
query = 'Who first described Alzheimer’s disease?'
rag_chain.invoke(query)

"Allois Alzheimer, a German psychiatrist and pathologist, first described Alzheimer's disease in 1906."

In [74]:
# 41. When was Alzheimer’s disease first described?
query = 'When was Alzheimer’s disease first described?'
rag_chain.invoke(query)

"Alzheimer's disease was first described by German psychiatrist and pathologist Alois Alzheimer in 1906."

In [75]:
# 42. What major research focus exists today?
query = 'What major research focus exists today?'
rag_chain.invoke(query)

"The major research focus today is to prevent Alzheimer's disease by intervening before the brain has been irreversibly damaged."

In [76]:
# 43. What is early-onset Alzheimer’s disease?
query = 'What is early-onset Alzheimer’s disease?'
rag_chain.invoke(query)

"Early-onset Alzheimer's disease is a neurodegenerative disease. \n\nIt is the most common form of dementia, accounting for around 60â€“70% of cases.\n\nThe most common early symptom is difficulty in remembering recent events.\n\nAs the disease advances, symptoms can include problems with language, disorientation (including easily getting lost), mood swings, loss of motivation, self-neglect, and behavioral issues.\n\nAs a person's condition declines, they often withdraw from family and society.\n\nPeople with Alzheimer's disease go through the reverse process of progressive cognitive impairment."

In [77]:
# 44. Is early-onset Alzheimer’s linked to genetics?
query = 'Is early-onset Alzheimer’s linked to genetics?'
rag_chain.invoke(query)

"Early-onset Alzheimer's is linked to genetics. \n\nWhen the disease is caused by autosomal dominant variants, it is known as early-onset familial Alzheimer's disease. \nEarly-onset Alzheimer's is about 90% heritable. \nAll autosomal dominant genetic causes of Alzheimer's disease affect either the amyloid precursor protein (APP) on chromosome 21 or the enzymes that generate AÎ², known as presenilin 1 and presenilin 2."

In [78]:
# 45. What is late-onset Alzheimer’s disease?
query = 'What is late-onset Alzheimer’s disease?'
rag_chain.invoke(query)

"There is no information about late-onset Alzheimer's disease in the provided context."

In [79]:
# 46. What complications can arise in advanced stages?
query = 'What complications can arise in advanced stages?'
rag_chain.invoke(query)

'In advanced stages, complications can arise particularly in the earliest stages of the disease. \n\nEating becomes problematic, requiring food to be prepared in smaller pieces or even puréed.\nSwallowing difficulties arise.'

In [80]:
# 47. What is a common cause of death in Alzheimer’s patients?
query = 'What is a common cause of death in Alzheimer’s patients?'
rag_chain.invoke(query)

"A common cause of death in Alzheimer's patients is usually an external factor, such as an infection of pressure ulcers or pneumonia."

In [81]:
# 48. What impact does Alzheimer’s disease have on caregivers?
query = 'What impact does Alzheimer’s disease have on caregivers?'
rag_chain.invoke(query)

"Caregiving for individuals with Alzheimer's disease must be carefully managed over the course of the disease since it has no cure and gradually renders people incapable of tending to their own needs.\n\nA caregiver's viewpoint is particularly important in the assessment of Alzheimer's disease, as they can supply important information on daily living abilities and on the decrease in the person's mental function.\n\nCaregivers play a crucial role in the differential diagnosis of Alzheimer's disease and other diseases through neurological examinations and interviews with family members.\n\nCaregiving can be challenging for families, often making it difficult for them to detect initial symptoms of Alzheimer's disease.\n\nAs a person's condition declines, they often withdraw from family and society, which can increase the burden on caregivers.\n\nCaregivers may experience emotional challenges, including managing mood swings, loss of motivation, and behavioral issues in the person with Alzh

In [82]:
# 49. Is Alzheimer’s disease a major public health issue?
query = 'Is Alzheimer’s disease a major public health issue?'
rag_chain.invoke(query)

"Yes, Alzheimer's disease is a major public health issue. \n\nIt is ranked as the seventh leading cause of death worldwide. \nIt has a large impact on society, with an estimated global annual cost of US$1 trillion. \nHalf of new dementia cases each year are Alzheimer's disease. \nThe disease-free population followed over the years have shown rates of 10 to 15 per thousand person-years for all dementias and 5–8 for Alzheimer's disease in Spain and Italy. \nAdvancing age is a primary risk factor for the disease, and incidence rates are not equal for all ages, with the risk approximately doubling every 5 years after the age of 65."

In [83]:
# 50. Why is Alzheimer’s disease considered a growing global concern?
query = 'Why is Alzheimer’s disease considered a growing global concern?'
rag_chain.invoke(query)

"Alzheimer's disease is considered a growing global concern due to its widespread impacts. \n\nThe disease has a large economic burden, with an estimated global annual cost of US$1 trillion.\n\nIt is a leading cause of death worldwide, ranked as the seventh leading cause of death.\n\nThe disease has a significant incidence rate, with rates between 10 and 15 per thousand person-years for all dementias and 5–8 for AD in certain countries.\n\nThe risk of acquiring the disease approximately doubles every 5 years after the age of 65, indicating that advancing age is a primary risk factor.\n\nThe large budget allocated for Alzheimer's research by health funders, such as the US National Institutes of Health's National Plan to Address Alzheimer's Disease with a budget of US$3.98 billion for fiscal year 2026, and the European Union's Horizon Europe research programme awarding over €570 million for dementia-related projects, further emphasizes its growing global concern."